# Train MixRec trên REES46 với WandB & Auto-Resume Checkpoint (Google Colab)
Notebook chạy trên Colab:
- Load dataset từ HuggingFace (nguyenmaiductrong/rees46-bpatmp-temporal).
- Tự động ghép file .npy thành Sparse Matrix của MixRec.
- Đánh giá Val (NDCG@20) lưu Best Model, Test Full-Ranking [10, 20, 50].
- Tự động lưu Checkpoint lên Google Drive.
- Logging bằng Weights & Biases (WandB).

### 1. Clone Source Code và Cài Đặt Môi Trường

In [ ]:
!git clone https://github.com/HKUDS/MixRec.git
!pip install datasets wandb scipy huggingface_hub

### 2. Tải trực tiếp các file .npy đã chia Train/Test từ HuggingFace
Đảm bảo tập Test của MixRec giống 100% với mô hình đang build.

In [ ]:
import os, json, pickle, numpy as np, scipy.sparse as sp
from huggingface_hub import hf_hub_download, login

repo_id = "nguyenmaiductrong/rees46-bpatmp-temporal"
repo_type = "dataset"

counts_file = hf_hub_download(repo_id, "node_counts.json", repo_type=repo_type)
with open(counts_file) as f:
    counts = json.load(f)
n_users = counts['user']
n_items = counts['product']

def build_sparse_from_hf(src_file, dst_file):
    src_path = hf_hub_download(repo_id, src_file, repo_type=repo_type)
    dst_path = hf_hub_download(repo_id, dst_file, repo_type=repo_type)
    rows = np.load(src_path)
    cols = np.load(dst_path)
    vals = np.ones(len(rows), dtype=np.float32)
    return sp.csr_matrix((vals, (rows, cols)), shape=(n_users, n_items))

trn_view = build_sparse_from_hf("view_train_src.npy", "view_train_dst.npy")
trn_cart = build_sparse_from_hf("cart_train_src.npy", "cart_train_dst.npy")
trn_buy  = build_sparse_from_hf("purchase_train_src.npy", "purchase_train_dst.npy")
tst_mat = build_sparse_from_hf("test_user_idx.npy", "test_product_idx.npy")
val_mat = build_sparse_from_hf("val_user_idx.npy", "val_product_idx.npy")

out_dir = "/content/MixRec/MixRec/Datasets/beibei"
os.makedirs(out_dir, exist_ok=True)

# PHẢI LƯU ĐÚNG TÊN FILE MẶC ĐỊNH CỦA BEIBEI ĐỂ GHI ĐÈ
pickle.dump(trn_view, open(f"{out_dir}/trn_pv", 'wb'))
pickle.dump(trn_cart, open(f"{out_dir}/trn_cart", 'wb'))
pickle.dump(trn_buy,  open(f"{out_dir}/trn_buy", 'wb'))
pickle.dump(tst_mat,  open(f"{out_dir}/tst_mat.pkl", 'wb'))
pickle.dump(val_mat,  open(f"{out_dir}/val_mat.pkl", 'wb'))

print("Hoan tat Data.")

### 3. Mount Google Drive và Login WandB

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import wandb
wandb.login()

### 4. Patch File Source Code Của MixRec (Val/Test Setup)

In [ ]:
!cd /content/MixRec/MixRec && git checkout mixrec_bei.py Utils/NNLayers.py DataHandler.py 2>/dev/null || true

import os
import re
import numpy as np

# === 1. VÁ DATAHANDLER.PY ===
dh_file = "/content/MixRec/MixRec/DataHandler.py"
if os.path.exists(dh_file):
    with open(dh_file, "r", encoding="utf-8") as f:
        dh_code = f.read()
    
    new_dh = """# val set
\t\twith open(self.predir + 'val_mat.pkl', 'rb') as fs:
\t\t\tval_mat = pickle.load(fs)
\t\tvalInt = np.array([val_mat.indices[val_mat.indptr[i]:val_mat.indptr[i+1]][0] if val_mat.indptr[i] < val_mat.indptr[i+1] else None for i in range(val_mat.shape[0])])
\t\tvalStat = (valInt != None)
\t\tself.valUsrs = np.reshape(np.argwhere(valStat!=False), [-1])
\t\tself.valInt = valInt

\t\t# test set
\t\twith open(self.predir + 'tst_mat.pkl', 'rb') as fs:
\t\t\ttst_mat = pickle.load(fs)
\t\ttstInt = np.array([tst_mat.indices[tst_mat.indptr[i]:tst_mat.indptr[i+1]][0] if tst_mat.indptr[i] < tst_mat.indptr[i+1] else None for i in range(tst_mat.shape[0])])
\t\ttstStat = (tstInt != None)
\t\tself.tstUsrs = np.reshape(np.argwhere(tstStat!=False), [-1])
\t\tself.tstInt = tstInt

\t\tself.trnMats = trnMats
\t\tself.label = trnMats[-1]
\t\tself.trnUsrs = trnUsrs"""
    dh_code = re.sub(r"# test set.*?self\.trnUsrs = trnUsrs", new_dh, dh_code, flags=re.DOTALL)
    with open(dh_file, "w", encoding="utf-8") as f:
        f.write(dh_code)
    print("Sua DataHandler.py thanh cong.")

# === 2. VÁ MIXREC_BEI.PY ===
file_path = "/content/MixRec/MixRec/mixrec_bei.py"
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        code = f.read()
        
    if "import tensorflow.compat.v1" not in code:
        code = code.replace("import tensorflow as tf", 
"""import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import wandb
import os
import numpy as np
""")

    if "from MixRec.DataHandler" in code:
        code = code.replace("from MixRec.DataHandler", "from DataHandler")

    new_run = """\tdef run(self):
\t\tself.prepareModel()
\t\tlog('Model Prepared')
\t\t
\t\tcheckpoint_dir = "/content/drive/MyDrive/MixRec_Checkpoints"
\t\twandb_id_file = f"{checkpoint_dir}/wandb_id.txt"
\t\timport os
\t\tos.makedirs(checkpoint_dir, exist_ok=True)
\t\tckpt = tf.train.latest_checkpoint(checkpoint_dir)
\t\tsaver = tf.train.Saver(max_to_keep=3)
\t\t
\t\tbest_val_ndcg = 0.0
\t\tbest_epoch = -1
\t\t
\t\tif ckpt:
\t\t\tprint(f"\n RESUMING CHECKPOINT: {ckpt}\n")
\t\t\tsaver.restore(self.sess, ckpt)
\t\t\tstloc = int(ckpt.split('-')[-1]) + 1
\t\t\tif os.path.exists(wandb_id_file):
\t\t\t\twith open(wandb_id_file, 'r') as f: run_id = f.read().strip()
\t\t\t\twandb.init(project="MixRec-REES46", resume="must", id=run_id)
\t\t\telse:
\t\t\t\trun = wandb.init(project="MixRec-REES46")
\t\t\t\twith open(wandb_id_file, 'w') as f: f.write(run.id)
\t\telif args.load_model != None:
\t\t\tself.loadModel()
\t\t\tstloc = len(self.metrics['TrainLoss']) * args.tstEpoch - (args.tstEpoch - 1)
\t\t\trun = wandb.init(project="MixRec-REES46")
\t\t\twith open(wandb_id_file, 'w') as f: f.write(run.id)
\t\telse:
\t\t\tstloc = 0
\t\t\tinit = tf.global_variables_initializer()
\t\t\tself.sess.run(init)
\t\t\tlog('Variables Inited')
\t\t\trun = wandb.init(project="MixRec-REES46")
\t\t\twith open(wandb_id_file, 'w') as f: f.write(run.id)
\t\t
\t\tfor ep in range(stloc, args.epoch):
\t\t\ttest = (ep % args.tstEpoch == 0)
\t\t\ttrain_reses = self.trainEpoch()
\t\t\tlog(self.makePrint('Train', ep, train_reses, test))
\t\t\t
\t\t\twandb_dict = {"Epoch": ep}
\t\t\tif isinstance(train_reses, dict):
\t\t\t\twandb_dict.update({f"Train_{k}": v for k, v in train_reses.items()})
\t\t\telse:
\t\t\t\twandb_dict["Train_Loss"] = train_reses
\t\t\t\t
\t\t\tif test:
\t\t\t\tval_reses = self.testEpoch(phase='val')
\t\t\t\tlog(self.makePrint('Val', ep, val_reses, test))
\t\t\t\tfor k, v in val_reses.items():
\t\t\t\t\tif isinstance(v, list) or isinstance(v, np.ndarray):
\t\t\t\t\t\twandb_dict[f"Val_{k}"] = v[0]
\t\t\t\t\telse:
\t\t\t\t\t\twandb_dict[f"Val_{k}"] = v
\t\t\t\t
\t\t\t\tcurrent_ndcg = val_reses.get('NDCG@20', 0)
\t\t\t\tif current_ndcg >= best_val_ndcg:
\t\t\t\t\tbest_val_ndcg = current_ndcg
\t\t\t\t\tbest_epoch = ep
\t\t\t\t\tsaver.save(self.sess, f"{checkpoint_dir}/best_model.ckpt")
\t\t\t\t\tprint(f"🌟 New Best Val NDCG@20: {best_val_ndcg:.4f} (Saved best_model.ckpt)")
\t\t\t
\t\t\twandb.log(wandb_dict)
\t\t\t
\t\t\tif ep % args.tstEpoch == 0:
\t\t\t\tself.saveHistory()
\t\t\t\tsaver.save(self.sess, f"{checkpoint_dir}/model.ckpt", global_step=ep)
\t\t\tprint()
\t\t
\t\tprint(f"Bắt đầu đánh giá TEST với Best Model từ Epoch {best_epoch}...")
\t\tbest_ckpt = f"{checkpoint_dir}/best_model.ckpt"
\t\tif os.path.exists(best_ckpt + ".index") or os.path.exists(best_ckpt + ".meta"):
\t\t\tsaver.restore(self.sess, best_ckpt)
\t\t
\t\t# Luôn log Test với k = [10, 20, 50]
\t\ttst_reses = self.testEpoch(phase='test')
\t\tlog(self.makePrint('Test', args.epoch, tst_reses, True))
\t\twandb_tst_dict = {}
\t\tfor k, v in tst_reses.items():
\t\t\twandb_tst_dict[f"Test_{k}"] = v
\t\twandb.log(wandb_tst_dict)
\t\tself.saveHistory()"""
    code = re.sub(r"\tdef run\(self\):.*?\t\tself\.saveHistory\(\)", new_run, code, flags=re.DOTALL)

    new_sampleTestBatch = """\tdef sampleTestBatch(self, batchIds, label, targetInt):
\t\tbatch = len(batchIds)
\t\ttemTst = targetInt[batchIds]
\t\ttemLabel = label[batchIds].toarray()
\t\tuIntLoc = np.repeat(batchIds, args.item)
\t\tiIntLoc = np.tile(np.arange(args.item), batch)
\t\treturn uIntLoc, iIntLoc, temTst, temLabel"""
    code = re.sub(r"\tdef sampleTestBatch\(self, batchIds, label, tstInt\):.*?\t\treturn uIntLoc, iIntLoc, temTst, tstLocs", new_sampleTestBatch, code, flags=re.DOTALL)

    new_testEpoch = """\tdef testEpoch(self, phase='val'):
\t\tif phase == 'val':
\t\t\tids = self.handler.valUsrs
\t\t\tif len(ids) > 20000:
\t\t\t\trng = np.random.RandomState(42)
\t\t\t\tids = rng.choice(ids, 20000, replace=False)
\t\t\ttargetInt = self.handler.valInt
\t\t\tk_list = [20]
\t\telse:
\t\t\tids = self.handler.tstUsrs
\t\t\ttargetInt = self.handler.tstInt
\t\t\tk_list = [10, 20, 50]
\t\tepochHits = {k: 0 for k in k_list}
\t\tepochNdcgs = {k: 0 for k in k_list}
\t\tnum = len(ids)
\t\ttstBat = 64
\t\tsteps = int(np.ceil(num / tstBat))
\t\tfeed_dict = {}
\t\tfor i in range(steps):
\t\t\tst = i * tstBat
\t\t\ted = min((i+1) * tstBat, num)
\t\t\tbatIds = ids[st: ed]
\t\t\tuLocs, iLocs, temTst, temLabel = self.sampleTestBatch(batIds, self.handler.label, targetInt)
\t\t\tfeed_dict[self.uids] = uLocs
\t\t\tfeed_dict[self.iids] = iLocs
\t\t\tfeed_dict[self.keepRate] = 1.0
\t\t\tpreds = self.sess.run(self.preds, feed_dict=feed_dict, options=tf.RunOptions(report_tensor_allocations_upon_oom=True))
\t\t\thits, ndcgs = self.calcRes(np.reshape(preds, [ed-st, args.item]), temTst, temLabel, k_list)
\t\t\tfor k in k_list:
\t\t\t\tepochHits[k] += hits[k]
\t\t\t\tepochNdcgs[k] += ndcgs[k]
\t\t\tlog('Steps %d/%d: %s evaluating...' % (i, steps, phase), save=False, oneline=True)
\t\tret = dict()
\t\tfor k in k_list:
\t\t\tret[f'HR@{k}'] = epochHits[k] / num
\t\t\tret[f'NDCG@{k}'] = epochNdcgs[k] / num
\t\treturn ret"""
    code = re.sub(r"\tdef testEpoch\(self\):.*?\t\treturn ret", new_testEpoch, code, flags=re.DOTALL)

    new_calcRes = """\tdef calcRes(self, preds, temTst, temLabel, k_list):
\t\thits = {k: 0 for k in k_list}
\t\tndcgs = {k: 0 for k in k_list}
\t\tfor j in range(preds.shape[0]):
\t\t\tpredvals = preds[j]
\t\t\tmask = temLabel[j]
\t\t\tpredvals[mask > 0] = -1e9 # Mask train items
\t\t\tsort_idx = np.argsort(-predvals)
\t\t\tfor k in k_list:
\t\t\t\tshoot = sort_idx[:k]
\t\t\t\tif temTst[j] in shoot:
\t\t\t\t\thits[k] += 1
\t\t\t\t\tndcgs[k] += np.reciprocal(np.log2(np.where(shoot == temTst[j])[0][0] + 2))
\t\treturn hits, ndcgs"""
    # Match original calcRes from github (return hit, ndcg)
    code = re.sub(r"\tdef calcRes\(self, preds, temTst, tstLocs\):.*?\t\treturn hit, ndcg", new_calcRes, code, flags=re.DOTALL)
    # Also match intermediate patches if they ran cell multiple times
    code = re.sub(r"\tdef calcRes\(self, preds, temTst, tstLocs\):.*?\t\treturn hits, ndcgs", new_calcRes, code, flags=re.DOTALL)
    code = re.sub(r"\tdef calcRes\(self, preds, temTst, temLabel\):.*?\t\treturn hits, ndcgs", new_calcRes, code, flags=re.DOTALL)
    
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(code)
    print("Sua mixrec_bei.py thanh cong.")

nn_file = "/content/MixRec/MixRec/Utils/NNLayers.py"
if os.path.exists(nn_file):
    with open(nn_file, "r", encoding="utf-8") as f:
        nn_code = f.read()
    if "from tensorflow.contrib.layers import xavier_initializer" in nn_code:
        nn_code = nn_code.replace("from tensorflow.contrib.layers import xavier_initializer", 
                                  "import tensorflow.compat.v1 as tf\nxavier_initializer = tf.initializers.glorot_normal")
        with open(nn_file, "w", encoding="utf-8") as f:
            f.write(nn_code)
        print("Sua NNLayers.py thanh cong.")
else:
    print(f"Khong tim thay {nn_file}")

### 5. Training!

In [ ]:
%%bash
cd /content/MixRec/MixRec
mkdir -p History Models

python mixrec_bei.py --data beibei --batch 1024